## Modules

In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import logging 
import sys
from agentz.pydantic_models import ExperimentConfiguration, ExecutedChunk
from agentz.utility import print_dict
from agentz import Agent
from pathlib import Path
import os
from agentz.utility import show_and_store_prepared_data, print_history, visualize_ui_elements, visualize_ui_elements2, show_transition
from agentz.memory import HistoryManager
import numpy as np
import matplotlib.pyplot as plt
from agentz.utility import show_graph, show_trim_output
import time
from agentz.pydantic_models import Episode


logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setLevel(logging.DEBUG)
if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
    logger.addHandler(stream_handler)
    
def load_settings(path : str) -> dict:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    with path.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}

    return ExperimentConfiguration(**cfg)  



## OSWorld Agent

In [ ]:
task = {
    "id": "94d95f96-9699-4208-98ba-3c3119edf9c2",
    "instruction": "I want to install Spotify on my current system. Could you please help me?",
    "config": [
        {
            "type": "execute",
            "parameters": {
                "command": [
                    "python",
                    "-c",
                    "import pyautogui; import time; pyautogui.click(960, 540); time.sleep(0.5);"
                ]
            }
        }
    ],
    "evaluator": {
        "func": "check_include_exclude",
        "result": {
            "type": "vm_command_line",
            "command": "which spotify"
        },
        "expected": {
            "type": "rule",
            "rules": {
                "include": ["spotify"],
                "exclude": ["not found"]
            }
        }
    }
}
 
SETTINGS_FILE = "../agent/config_files/start_agent_nb.yaml"
settings = load_settings(SETTINGS_FILE)
agent = await Agent.create("my-agent", settings)
#agent.run_task_bdi(task, verbose = True, max_cycles=5)

## Agent control loop

In [ ]:
from agentz.perception import PerceptionInterface

SETTINGS_FILE = "../agent/config_files/start_agent_nb.yaml"
settings = load_settings(SETTINGS_FILE)
agent.perception = PerceptionInterface(
     settings.perception_settings,
     parallel=True,
     debug_visualizations=True
)
perception = agent.env.observe()
last_observation = agent.perception.process(perception)
agent.history_manager.update(last_observation, tags=["observation"])

df = visualize_ui_elements(
    ui_dict=last_observation.ui_elements,
    screenshot=last_observation.screenshot,
    show_label=True,
    include_label_text=False,
    title="Initial UI Observation",
    text_color="#000000"
)

In [ ]:
last_observation.a11y_raw

In [ ]:
last_observation.a11y_raw

In [ ]:
a = agent.planner.propose_next_steps(
    task=task,
    history_manager=agent.history_manager,
    system_info=vm_info,
    memory = agent.memory_manager,
    tms = agent.tms
    )


print(a)

#### Initialization

In [ ]:
verbose = True

agent._last_task = task
agent.logger.info("Starting new task execution")
agent.logger.info("Task:\n%s", task)

# Reset environment and probe system
perception = agent.env.reset(task)
vm_info = agent.tools.os_inspector.probe()

# Episode bookkeeping
episode = Episode(
    episode_id=os.urandom(16).hex(),
    task=task,
    instruction=task.get("instruction"),
    status="STARTED",
    started_ts_ms=time.time_ns() // 1_000_000,
    os_name=vm_info["os"]["pretty_name"],
    desktop_env=vm_info.get("desktop_environment", "unknown"),
    display_server=vm_info.get("display_server", "unknown"),
)

agent.logger.info(
    "Episode started | episode_id=%s | task_id=%s",
    episode.episode_id,
    task.get("id"),
)

# --------------------------------------------------
# Initial observation (beliefs bootstrap)
# --------------------------------------------------

last_observation = agent.perception.process(perception)
agent.history_manager.update(last_observation, tags=["observation"])

if verbose:
    visualize_ui_elements(
        ui_dict=agent.history_manager.last_observation.ui_elements,
        screenshot=agent.history_manager.last_observation.screenshot,
        show_label=True,
        include_label_text=False,
        title="Initial UI Observation",
        text_color="#000000"

    )
    

#### Main loop

In [ ]:
max_cycles = 1
verbose = False

for cycle in range(int(max_cycles)):
    agent.logger.info("=== Cycle %d / %d ===", cycle + 1, max_cycles)

    # ----------------------------
    # Planning (Desires)
    # ----------------------------

    action_chunk = agent.planner.propose_next_steps(
        task=task,
        history_manager=agent.history_manager,
        system_info=vm_info,
        memory = agent.memory_manager,
        tms = agent.tms
    )

    agent.history_manager.update(action_chunk, tags=["start_chunk"])

    agent.logger.info(
        "Planner produced chunk | macro_goal='%s' | decision=%s | steps=%d",
        action_chunk.macro_goal,
        action_chunk.decision,
        len(action_chunk.steps),
    )

    if action_chunk.decision in ("DONE", "FAIL"):
        agent.logger.info("Planner decided to %s. Ending loop.", action_chunk.decision)
        break

    # ----------------------------
    # Execution (Intentions)
    # ----------------------------

    for k, step in enumerate(action_chunk.steps):
        agent.history_manager.update(step, tags=["step"])

        agent.logger.debug(
            "Executing step %d | %s | type=%s | pause=%.2f",
            k,
            step.description,
            step.action_type,
            step.pause,
        )

        perception = agent.executor.execute_step(step)
        last_observation = agent.perception.process(perception)
        
        agent.history_manager.update(last_observation, tags=["observation", "observation_after_step"],)

        if verbose:
            visualize_ui_elements(
                ui_dict=agent.history_manager.last_observation.ui_elements,
                screenshot=agent.history_manager.last_observation.screenshot,
                show_label=True,
                include_label_text=False,
                title=f"Observation after step {k + 1}/{len(action_chunk.steps)}",
                text_color="#000000"

            )

    # ----------------------------
    # Evaluation (Judge)
    # ----------------------------

    agent.logger.info("Evaluating executed chunk")
    last_evaluation = agent.judge.evaluate_outcome(agent.history_manager)
    
    agent.history_manager.update({"chunk": action_chunk, "evaluation": last_evaluation}, tags=["end_chunk"])
    # Compact evaluation log
    agent._log_chunk_evaluation(agent.history_manager.last_chunk)

    # ----------------------------
    # Memory update (TRIM + TMS)
    # ----------------------------

    agent.logger.info("Running TRIM")
    trim_out = agent.trim.run(
        task_instruction=task.get("instruction", ""),
        tms_nodes=agent.tms.nodes(),
        history_manager = agent.history_manager,
        current_observation=agent.history_manager.last_observation,
        chunk_digest=agent.history_manager.last_chunk_digest_for_tms(),
        cid=episode.episode_id,
    )
    agent.history_manager.update(trim_out, tags=["trim_info"])
    
    agent.logger.info("Updating TMS")
    agent.tms.apply_trim_output(
        trim_out,
        agent.history_manager.last_chunk,
        agent.history_manager.last_observation,
    )

    agent.logger.info("Cycle %d completed\n", cycle + 1)

# --------------------------------------------------
# Final evaluation
# --------------------------------------------------

agent.logger.info("Agent finished execution. Evaluating final result.")
osworld_score = agent.env.evaluate()

episode.score = {
    "status": osworld_score["status"],
    "metric": osworld_score["result"]["metric"],
    "success": osworld_score["result"]["success"],
    "stats" : None
}

episode.finished_ts_ms = time.time_ns() // 1_000_000
episode.status = "DONE" if episode.score['success'] else "FAIL"

from agentz.memory.utils import append_metrics_csv

# Compute and attach metrics (history-based)
episode.score["stats"] = agent.history_manager.compute_metrics(episode)
append_metrics_csv(path = "../data/experiments.csv", episode = episode)

agent.memory_manager.ingest_end_of_episode(
    episode=episode,
    history_manager=agent.history_manager,
    tms=agent.tms
)

agent.logger.info("Final score: %s", episode.score)


In [ ]:
from typing import Dict, Any, List, Optional


def visualize_episode(*, episode, history, tms) -> Dict[str, str]:
    """
    Build a human-readable visualization of:
    - Episode metadata
    - Execution trace (chunks + steps + failures)
    - TMS structure (tree + detailed per-node blocks + table)

    Returns a dict of named text artifacts.
    """

    artifacts: Dict[str, str] = {}

    # ---------------------------
    # Helpers
    # ---------------------------

    def _clip(s: Optional[str], n: int = 110) -> str:
        if not s:
            return ""
        s = " ".join(str(s).split())
        return s if len(s) <= n else s[: n - 1] + "…"

    def _fmt_bool(b: Optional[bool]) -> str:
        if b is True:
            return "T"
        if b is False:
            return "F"
        return "-"

    def _fmt_status(x) -> str:
        # Supports enums with .name or plain strings
        return getattr(x, "name", str(x))

    def _fmt_anchor(a) -> str:
        # Expected anchor fields: label, role, qx, qy
        label = getattr(a, "label", "") or ""
        role = getattr(a, "role", "") or ""
        qx = getattr(a, "qx", None)
        qy = getattr(a, "qy", None)
        lr = f"{label}|{role}".strip("|")
        if qx is None or qy is None:
            return lr or "<anchor>"
        return f"{lr}|q({qx},{qy})"

    def _node_mark(node) -> str:
        # Prefer node.last_success if present
        success = getattr(node, "last_success", None)
        if success is True:
            return "✔"
        if success is False:
            return "✖"
        return "·"

    # ============================================================
    # EPISODE HEADER
    # ============================================================

    header_lines: List[str] = []
    header_lines.append("EPISODE SUMMARY")
    header_lines.append("=" * 100)
    header_lines.append(f"Episode ID     : {episode.episode_id}")
    header_lines.append(f"Task           : {_clip(episode.instruction, 160)}")
    header_lines.append(f"Status         : {episode.status}")
    header_lines.append(f"OS             : {episode.os_name}")
    header_lines.append(f"Desktop        : {episode.desktop_env}")
    header_lines.append(f"Display Server : {episode.display_server}")

    if episode.finished_ts_ms:
        duration = episode.finished_ts_ms - episode.started_ts_ms
        header_lines.append(f"Duration (ms)  : {duration}")

    artifacts["episode_header"] = "\n".join(header_lines)

    # ============================================================
    # EXECUTION TRACE (CHUNKS + STEPS)
    # ============================================================

    trace_lines: List[str] = []
    trace_lines.append("EXECUTION TRACE")
    trace_lines.append("=" * 100)

    if not getattr(history, "chunks_history", None):
        trace_lines.append("(no chunks executed)")
    else:
        for ci, chunk in enumerate(history.chunks_history, start=1):
            trace_lines.append("")
            trace_lines.append(f"CHUNK {ci}")
            trace_lines.append("-" * 100)
            trace_lines.append(f"Macro-goal      : {_clip(chunk.macro_goal, 140)}")
            trace_lines.append(f"Decision        : {chunk.decision}")
            trace_lines.append(f"Overall success : {_fmt_bool(chunk.overall_success)}")
            trace_lines.append(f"Failure type    : {chunk.failure_type}")
            trace_lines.append(f"Failing step    : {chunk.failing_step_index}")
            trace_lines.append(f"Post state      : {_clip(chunk.post_chunk_state, 180)}")
            if getattr(chunk, "planner_guidance", None):
                trace_lines.append(f"Planner guidance: {_clip(chunk.planner_guidance, 180)}")

            trace_lines.append("")
            trace_lines.append("Steps:")
            for step, eval_ in zip(chunk.steps, chunk.steps_eval):
                mark = "✔" if eval_.success else "✖"
                trace_lines.append(f"  [{mark}] Step {step.index}: {_clip(step.description, 140)}")
                trace_lines.append(f"       expected : {_clip(step.expected_outcome, 160)}")
                trace_lines.append(f"       evidence : {_clip(eval_.evidence, 180)}")
                if not eval_.success:
                    if getattr(eval_, "failure_reason", None):
                        trace_lines.append(f"       reason   : {_clip(eval_.failure_reason, 160)}")
                    if getattr(eval_, "fix_suggestion", None):
                        trace_lines.append(f"       fix      : {_clip(eval_.fix_suggestion, 160)}")

    artifacts["execution_trace"] = "\n".join(trace_lines)

    # ============================================================
    # TMS TREE VIEW (NO GRAPHVIZ)
    # ============================================================

    def _build_adjacency():
        children: Dict[str, List[str]] = {}
        parents: set[str] = set()
        for e in tms.edges():
            children.setdefault(e.parent_id, []).append(e.child_id)
            parents.add(e.child_id)

        # Keep deterministic order
        for k in children:
            children[k] = sorted(children[k])

        roots = sorted([n.node_id for n in tms.nodes() if n.node_id not in parents])
        return roots, children

    def _render_node(node_id: str, prefix: str = "", is_last: bool = True) -> List[str]:
        node = tms.get_node(node_id)
        if not node:
            return []

        branch = "└─ " if is_last else "├─ "
        status = _fmt_status(node.status)
        lines = [f"{prefix}{branch}{_node_mark(node)} {node.title} [{status}] ({node.node_id})"]

        # Keep the tree compact: only show a clipped value and quick metadata
        if getattr(node, "value", None):
            lines.append(f"{prefix}   value: {_clip(node.value, 140)}")

        if getattr(node, "last_outcome", None):
            lines.append(f"{prefix}   last_outcome: {_clip(node.last_outcome, 140)}")

        if getattr(node, "last_guidance", None):
            lines.append(f"{prefix}   last_guidance: {_clip(node.last_guidance, 140)}")

        revs = getattr(node, "revisions", None) or []
        anchors = getattr(node, "anchors", None) or []
        upd = getattr(node, "last_updated_step", None)
        lines.append(f"{prefix}   meta: rev={len(revs)} anc={len(anchors)} last_step={upd}")

        kids = children_map.get(node_id, [])
        for i, child in enumerate(kids):
            last = i == len(kids) - 1
            ext = "   " if is_last else "│  "
            lines.extend(_render_node(child, prefix + ext, last))

        return lines

    tree_lines: List[str] = []
    tree_lines.append("TASK MEMORY STRUCTURE (TREE)")
    tree_lines.append("=" * 100)

    roots, children_map = _build_adjacency()
    if not roots:
        tree_lines.append("(empty TMS)")
    else:
        for r in roots:
            tree_lines.extend(_render_node(r))
            tree_lines.append("")

    artifacts["tms_tree"] = "\n".join(tree_lines).rstrip()

    # ============================================================
    # TMS NODE DETAILS (PRINTABLE BLOCKS)
    # ============================================================

    details_lines: List[str] = []
    details_lines.append("TASK MEMORY STRUCTURE (NODE DETAILS)")
    details_lines.append("=" * 100)

    # Sort nodes by title for readability
    nodes_sorted = sorted(list(tms.nodes()), key=lambda n: (getattr(n, "title", ""), getattr(n, "node_id", "")))
    for n in nodes_sorted:
        status = _fmt_status(n.status)
        details_lines.append("")
        details_lines.append(f"[{n.node_id}] [{status}]")
        details_lines.append(f"title={n.title}")
        details_lines.append(f"value={_clip(getattr(n, 'value', ''), 220)}")

        revs = getattr(n, "revisions", None) or []
        anchors = getattr(n, "anchors", None) or []
        details_lines.append(f"rev={len(revs)} anc={len(anchors)} last_step={getattr(n, 'last_updated_step', None)}")

        # Revisions: show a compact timeline (rev_id:summary)
        if revs:
            rev_items = []
            for r in revs:
                rid = getattr(r, "rev_id", "?")
                summ = getattr(r, "summary", None) or ""
                if not summ:
                    # Fallback: tag based on value presence
                    summ = "update"
                rev_items.append(f"r{rid}:{summ}")
            details_lines.append(f"revisions=[{', '.join(rev_items[:12])}{'…' if len(rev_items) > 12 else ''}]")
        else:
            details_lines.append("revisions=[]")

        # Anchors: show up to N anchors as "label|role|q(x,y)"
        if anchors:
            anc_items = [_fmt_anchor(a) for a in anchors]
            details_lines.append(f"anchors=[{', '.join(anc_items[:10])}{'…' if len(anc_items) > 10 else ''}]")
        else:
            details_lines.append("anchors=[]")

        # Optional judge-derived metadata if present
        if getattr(n, "last_success", None) is not None:
            details_lines.append(f"last_success={getattr(n, 'last_success')}")
        if getattr(n, "last_outcome", None):
            details_lines.append(f"last_outcome={_clip(getattr(n, 'last_outcome'), 220)}")
        if getattr(n, "last_guidance", None):
            details_lines.append(f"last_guidance={_clip(getattr(n, 'last_guidance'), 220)}")

    artifacts["tms_node_details"] = "\n".join(details_lines).rstrip()

    # ============================================================
    # TMS TABLE VIEW (COMPACT)
    # ============================================================

    table_lines: List[str] = []
    table_lines.append("TASK MEMORY STRUCTURE (TABLE)")
    table_lines.append("=" * 100)
    table_lines.append(
        f"{'ID':<22} {'STATUS':<10} {'REV':>3} {'ANC':>3} {'SUC':>3} {'LAST':>5}  TITLE / VALUE"
    )
    table_lines.append("-" * 100)

    for n in nodes_sorted:
        nid = n.node_id
        status = _fmt_status(n.status)
        revs = getattr(n, "revisions", None) or []
        anchors = getattr(n, "anchors", None) or []
        suc = getattr(n, "last_success", None)
        last_step = getattr(n, "last_updated_step", None)
        title = _clip(getattr(n, "title", ""), 44)
        value = _clip(getattr(n, "value", ""), 54)
        table_lines.append(
            f"{nid:<22} {status:<10} {len(revs):>3} {len(anchors):>3} {_fmt_bool(suc):>3} "
            f"{str(last_step):>5}  {title} | {value}"
        )

    artifacts["tms_table"] = "\n".join(table_lines)

    return artifacts


# Example usage
artifacts = visualize_episode(
    episode=episode,
    history=agent.history_manager,
    tms=agent.tms,
)

print(artifacts["episode_header"])
print()
print(artifacts["execution_trace"])
print()
print(artifacts["tms_tree"])
print()
print(artifacts["tms_node_details"])
print()
print(artifacts["tms_table"])